# EXTRACTION OF RASTERS BY AREA

- Raster reading: Opened only once in read mode.

- By zone: Loading of the limits as a shapefile

- Precise masking: We use geometry_mask to count only the pixels in the polygon.

- Result: Centre coordinates + value, saved in a CSV file per zone.

- RAM management: We only read the small window of the raster corresponding to each polygon, avoiding complete reading.

In [ ]:
# installing the python library to read and interact with rasters
!pip install rasterio

In [ ]:
# connection to the drive where the rasters are kept and where the results will be saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# importing all libraries needed for the extraction
import rasterio
import rasterio.mask
import geopandas as gpd
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import re

In [ ]:
# Here the parameters for the code to run smoothly are defined
# this code will loop on all the grids saved in a zipped folder (less place used and still efficient)

# === PARAMETERS ===
# path in the drive to the raster
raster_path = "/content/drive/****.tif"
# value of the raster that will be extracted
value = "classGLWD"
# path to the shapefile that defines each zone (level 2 of watersheds)
shapefile_path = "/content/drive/****.shp"
# ID field that identifies the different zone
ID = "PFAF_ID"
# path to the folder where you want the results to be saved as .csv in your drive
OUTPUT_FOLDER = "/content/drive/output"

# if the output folder does not exist, it is created
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# Loading data
gdf = gpd.read_file(shapefile_path)
print("✅ Limits shapefile loaded.")

# === Reading the raster
print("⏳ Opening the raster...")
with rasterio.open(raster_path) as src:

    # Extract the zone's number from the name of the file
    match = re.search(r'Zone_(\d+)', shapefile_path)  # rename the search depending on the filename format
    if match:
        tile_id = match.group(1)
    else:
        print(f"Impossible to find the zone's number in this file : {shapefile_path}")
        tile_id = None

    # get the geometry
    shapes = gdf.geometry.values

    # crop the raster and transformation
    out_image, out_transform = rasterio.mask.mask(src, shapes, crop=True)

    # extract value of raster
    extract = out_image[0].astype(float)
    # transform no-value data to NaN
    extract[extract == 255] = np.nan
    # extract height and with of raster
    height, width = extract.shape

    # creating the columns and rows based on width and height
    rows, cols = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")

    # get the coordinates for each cell/pixel of the raster
    xs, ys = rasterio.transform.xy(out_transform, rows, cols)
    # organising them as lat and long
    lons = np.array(xs)
    lats = np.array(ys)

    # get results
    data = {
        "latitude": lats.flatten(),
        "longitude": lons.flatten(),
        f"{value}": extract.flatten()
    }
    # create the dataframe of the results
    df_exact = pd.DataFrame(data)

    # Save as .csv
    try:
        # create a double entry matrix based on lat and long
        pivoted = df_exact.pivot(index="latitude", columns="longitude", values=f"{value}")
        # keep the lat and long in the right order (N-S and W-E)
        pivoted = pivoted.sort_index(ascending=False)
        # save as csv
        OUTPUT_CSV = os.path.join(OUTPUT_FOLDER, f"Pivot_{value}_zone_{tile_id}.csv")
        pivoted.to_csv(OUTPUT_CSV)
        print(f"\n✅ Double entry table saved at : {OUTPUT_CSV}")
    except Exception as e:
        # in case there was a problem to create the double entry table, the dataframe will be saved as is
        output_csv = os.path.join(OUTPUT_FOLDER, f"Extraction_{value}_zone_{tile_id}.csv")
        df_exact.to_csv(output_csv, index=False)
        print(f"\n❌ Double entry was not created : {e}\n✅ Data still saved in brut at {output_csv}.")